In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import utulek

utulek.platform.import_globals_notebook(globals())

base_module = utulek
IS_NOTEBOOK_KERNEL_CODE = True
NOTEBOOK_NAME = 2026_07_27-lllm.2.ipynb
ASSET_PATH = /home/gilgamesh/main.syncthing/utulek/experiment/2026_07_27-lllm.ipynb.asset/
Loaded `/home/gilgamesh/main.syncthing/utulek/experiment/.env`.
torch: ['NVIDIA GeForce GTX 1080 Ti']
tf: /device:GPU:0
jax: [CudaDevice(id=0)]


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint,
	dtype="auto",
	device_map="auto")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [4]:
tokenizer.response_schema = {
	"x-regex":
	r"^(?:(?:<think>)?\s*(?P<thinking>.+?)\s*</think>)?\s*(?:<tool_call>(?P<tool_calls>.*?)\s*</tool_call>)?\s*(?P<content>.+?)?\s*(?:<\|im_end\|>|$)",
	"type": "object",
	"properties": {
	"role": {
	"const": "assistant"
	},
	"content": {
	"type": "string"
	},
	"thinking": {
	"type": "string"
	},
	"tool_calls": {
	"x-regex-iterator": r"^(.*)$",
	"type": "array",
	"items": {
	"type": "object",
	"properties": {
	"type": {
	"const": "function"
	},
	"function": {
	"x-parser": "json",
	"x-parser-args": {
	"allow_non_json": True
	},
	"type": "object",
	"properties": {
	"name": {
	"type": "string"
	},
	"arguments": {
	"type": "object",
	"additionalProperties": {}
	},
	},
	},
	},
	},
	},
	},
}

In [5]:
def prompt(user_prompt: str,
	system_prompt:
	str = "You are a general-purpose assistant bot.",
	max_new_tokens: int = 2048):
	messages = [{
		"role": "system",
		"content": system_prompt
	}, {
		"role": "user",
		"content": user_prompt
	}]
	input_ids = tokenizer.apply_chat_template(messages,
		add_generation_prompt=True,
		return_dict=True,
		return_tensors="pt")["input_ids"].to(model.device)
	outputs = model.generate(input_ids,
		max_new_tokens=max_new_tokens)
	out_text = tokenizer.decode(
		outputs[0][len(input_ids[0]):])
	res = tokenizer.parse_response(out_text)
	return res

In [6]:
res = prompt("Explain Burnside’s Lemma.",
	max_new_tokens=2048)
print(res["content"])

**Burnside's Lemma** (also known as the **Cauchy-Frobenius Lemma**) is a fundamental theorem in combinatorics and group theory. It provides a way to count the number of distinct objects when symmetries (like rotations or reflections) are taken into account.

In simpler terms: **It tells you how many unique ways you can arrange things if some arrangements are considered "the same" because they can be transformed into each other by symmetry.**

---

### 1. The Problem: Counting Orbits
Imagine you have a set of items $X$ (like colored beads) and a group of symmetries $G$ (like rotating the necklace).
*   If you rotate a necklace, the physical arrangement changes, but the *pattern* remains the same.
*   We want to count the number of **distinct patterns** (orbits), not the total number of raw arrangements.

**Example:**
Suppose you have a necklace with **3 beads**. You can color each bead either **Red** or **Blue**.
*   **Total raw colorings:** $2^3 = 8$.
*   **Symmetry:** You can rotate t

In [7]:
print(res["thinking"])

Here's a thinking process that leads to the explanation of Burnside's Lemma:

1.  **Deconstruct the Request:**
    *   **Topic:** Burnside's Lemma (also known as the Cauchy-Frobenius Lemma).
    *   **Goal:** Explain it clearly.
    *   **Target Audience:** Likely someone interested in combinatorics, group theory, or discrete mathematics. The explanation should be accessible but mathematically precise.

2.  **Initial Brainstorming & Key Concepts:**
    *   What is it? A formula for counting distinct objects under symmetry (group actions).
    *   What is the core problem? "How many ways can I color a necklace with $n$ beads using $k$ colors, if rotations are considered the same?"
    *   What are the key terms?
        *   Set $X$: The objects being colored/arranged.
        *   Group $G$: The symmetries (rotations, reflections).
        *   Action: How $G$ acts on $X$.
        *   Orbit: A set of equivalent objects.
        *   Fixed Point: An object unchanged by a specific group elem

In [8]:
res = prompt("Explain Burnside’s Lemma.",
	max_new_tokens=8192)
print(res["content"])

KeyboardInterrupt: 

In [ ]:
from transformers import TextIteratorStreamer
from threading import Thread


def prompt(user_prompt: str,
	history=[{
	"role":
	"system",
	"content":
	"You are a general-purpose assistant bot. Don't overthink."
	}],
	max_new_tokens: int = 2048,
	enable_thinking: bool = True):
	messages = history
	messages.append({"role": "user", "content": user_prompt})
	input_ids = tokenizer.apply_chat_template(messages,
		add_generation_prompt=True,
		return_dict=True,
		enable_thinking=enable_thinking,
		return_tensors="pt")["input_ids"].to(model.device)
	streamer = TextIteratorStreamer(tokenizer,
		skip_prompt=True,
		skip_special_tokens=True)
	generation_args = {
		"input_ids": input_ids,
		"max_new_tokens": max_new_tokens,
		"temperature": 0.6,
		"top_p": 0.95,
		"top_k": 20,
		"min_p": 0,
		"do_sample": True,
		"streamer": streamer,
	}
	thread = Thread(target=model.generate,
		kwargs=generation_args)
	thread.start()
	tokens = []
	for token in streamer:
		tokens.append(token)
		yield token
	thread.join()
	history.append({
		"role": "assistant",
		"content": "".join(tokens)
	})

In [36]:
history = [{
	"role":
	"system",
	"content":
	"You are a general-purpose assistant bot."
}]
for token in prompt("Give me a number.",
	history=history,
	enable_thinking=False):
	print(token, end="")
pp(history)

42
[{'role': 'system', 'content': 'You are a general-purpose assistant bot.'},
 {'role': 'user', 'content': 'Give me a number.'},
 {'role': 'assistant', 'content': '42\n'}]


In [37]:
for token in prompt("Give me another number.",
	history=history,
	enable_thinking=False):
	print(token, end="")
pp(history)

7
[{'role': 'system', 'content': 'You are a general-purpose assistant bot.'},
 {'role': 'user', 'content': 'Give me a number.'},
 {'role': 'assistant', 'content': '42\n'},
 {'role': 'user', 'content': 'Give me another number.'},
 {'role': 'assistant', 'content': '7\n'}]


In [38]:
for token in prompt(
		"Multiply those previous numbers together.",
		history=history,
		enable_thinking=False):
	print(token, end="")
pp(history)

$42 \times 7 = 294$
[{'role': 'system', 'content': 'You are a general-purpose assistant bot.'},
 {'role': 'user', 'content': 'Give me a number.'},
 {'role': 'assistant', 'content': '42\n'},
 {'role': 'user', 'content': 'Give me another number.'},
 {'role': 'assistant', 'content': '7\n'},
 {'role': 'user', 'content': 'Multiply those previous numbers together.'},
 {'role': 'assistant', 'content': '$42 \\times 7 = 294$\n'}]


In [30]:
for token in prompt(
		"Explain Burnside’s Lemma. I am a first year undergraduate math student.",
		history=history,
		max_new_tokens=8192,
		enable_thinking=False):
	print(token, end="")

Hello! It is great to meet a first-year undergraduate student. Burnside's Lemma is a beautiful and powerful tool in combinatorics that connects **group theory** with **counting**.

Here is a step-by-step explanation tailored to your level.

### 1. The Core Problem: Why do we need it?
Imagine you have a set of objects (like colored beads on a necklace or tiles on a floor) and a group of symmetries (like rotating the necklace or flipping the tiles).

*   **Naive Approach:** You might try to list every possible arrangement and divide by the number of symmetries.
*   **The Problem:** This fails if two different arrangements look the same after a symmetry (they are "equivalent"). Counting them directly is messy.

**Burnside's Lemma** gives us a formula to count the number of **distinct orbits** (unique arrangements) without listing them all. It relies on a simple idea: **Count how many things stay the same under each symmetry.**

---

### 2. The Definitions You Need

Before the formula, let

KeyboardInterrupt: 

In [45]:
history = [{
	"role":
	"system",
	"content":
	"You are a general-purpose assistant bot. Don't overthink."
}]
for token in prompt("Give me a number.",
	history=history,
	enable_thinking=True):
	print(token, end="")
pp(history)

Thinking Process:

1.  **Analyze the Request:** The user is asking for "a number". This is a very open-ended request.

2.  **Determine the Intent:** The user might want a random number, a specific type of number (like a phone number, ID, etc.), or just a simple numerical value for fun. Since no context is given, the safest bet is to provide a random, reasonable number.

3.  **Select a Number:**
    *   Option 1: A single digit (1-9).
    *   Option 2: A two-digit number (10-99).
    *   Option 3: A larger number (like 42, the Hitchhiker's Guide to the Galaxy answer).
    *   Option 4: A decimal.
    *   Option 5: A date.
    *   Option 6: A timestamp.

    Most likely, they want something simple and random. Let's go with a random integer between 1 and 100, or perhaps the famous "42". Or just a random one like 7.

    Let's choose a random positive integer. 43.

4.  **Formulate the Response:** Keep it simple and direct. "43" or "Here's a number: 43".

5.  **Refine:** Just giving the num

In [46]:
for token in prompt("Give me another number.",
	history=history,
	enable_thinking=True):
	print(token, end="")
pp(history)

Thinking Process:

1.  **Analyze the Request:** The user is asking for another number, implying they want a different one than the previous one I provided.
2.  **Recall Previous Interaction:** In the previous turn, I provided the number "42" (after some internal deliberation).
3.  **Determine Intent:** The user wants a new, random number.
4.  **Select a Number:** I need to pick a number different from 42.
    *   Options: 1, 7, 43, 99, 100, etc.
    *   Let's pick something simple and distinct. Maybe 7 was a good candidate, but I settled on 42.
    *   Let's pick 7 this time. Or 13. Or 99.
    *   Let's go with 7.
5.  **Formulate the Response:** Keep it consistent with the previous tone (simple, direct).
    *   "7" or "Here's another number: 7."
6.  **Refine:** Just "7" is fine. Or "7".
    *   Let's make it slightly conversational to show I'm responding to the "another" part.
    *   "7" is good.
    *   "How about 7?"
    *   "Here's one: 7."
    *   Let's just give the number. "7".

KeyboardInterrupt: 

In [ ]:
for token in prompt(
		"Multiply those previous numbers together.",
		history=history,
		enable_thinking=True):
	print(token, end="")
pp(history)

$42 \times 7 = 294$
[{'role': 'system',
  'content': "You are a general-purpose assistant bot. Don't overthink."},
 {'role': 'user', 'content': 'Give me a number.'},
 {'role': 'assistant',
  'content': 'Thinking Process:\n'
             '\n'
             '1.  **Analyze the Request:** The user is asking for "a number". '
             'This is a very open-ended request.\n'
             '2.  **Determine Constraints:** There are no specific constraints '
             '(e.g., "a prime number", "a number between 1 and 10", "your '
             'favorite number").\n'
             '3.  **Select a Number:** I need to pick a number that is '
             'reasonable, perhaps random or interesting.\n'
             '    *   Option 1: 42 (The "Answer to the Ultimate Question of '
             'Life, the Universe, and Everything").\n'
             '    *   Option 2: 7 (Lucky number).\n'
             '    *   Option 3: A random integer like 12345.\n'
             '    *   Option 4: Just a simple numb

In [49]:
from transformers import TextIteratorStreamer
from threading import Thread


def prompt(user_prompt: str,
	history=[{
	"role":
	"system",
	"content":
	"You are a general-purpose assistant bot. Don't overthink."
	}],
	max_new_tokens: int = 2048,
	enable_thinking: bool = True):
	messages = history
	messages.append({"role": "user", "content": user_prompt})
	input_ids = tokenizer.apply_chat_template(messages,
		add_generation_prompt=True,
		return_dict=True,
		enable_thinking=enable_thinking,
		return_tensors="pt")["input_ids"].to(model.device)
	streamer = TextIteratorStreamer(tokenizer,
		skip_prompt=True,
		skip_special_tokens=True)
	generation_args = {
		"input_ids": input_ids,
		"max_new_tokens": max_new_tokens,
		"temperature": 0.6,
		"top_p": 0.95,
		"top_k": 20,
		"min_p": 0.0,
		# "presence_penalty": 0.0,
		"repetition_penalty": 1.0,
		"do_sample": True,
		"streamer": streamer,
	}
	thread = Thread(target=model.generate,
		kwargs=generation_args)
	thread.start()
	tokens = []
	for token in streamer:
		tokens.append(token)
		yield token
	thread.join()
	history.append({
		"role": "assistant",
		"content": "".join(tokens)
	})

In [50]:
history = [{
	"role":
	"system",
	"content":
	"You are a general-purpose assistant bot. Don't overthink."
}]
for token in prompt("Give me a number.",
	history=history,
	enable_thinking=True):
	print(token, end="")
pp(history)

Thinking Process:

1.  **Analyze the Request:**
    *   User: "Give me a number."
    *   Intent: The user wants a single numerical value.
    *   Constraints: None specified (e.g., range, type, context).

2.  **Determine the Output:**
    *   A simple number is sufficient.
    *   To be engaging but not overcomplicated, I can pick a random number (like 42, 7, 123, etc.).
    *   Since the prompt is very open-ended, I'll just provide a straightforward number. Maybe add a little context or just the number itself.
    *   Let's go with a simple, neutral number like 42 (The Answer to Life, the Universe, and Everything) or something random like 7.
    *   Actually, just giving a number is the most direct response. Let's pick 42 for fun, or maybe a random integer like 88.
    *   Decision: Provide a single number.

3.  **Draft Response:**
    *   "42"
    *   "7"
    *   "123"
    *   "Let's go with 42."

4.  **Refine Response:**
    *   Keep it concise.
    *   "42" is a classic.
    *   "

In [51]:
for token in prompt("Give me another number.",
	history=history,
	enable_thinking=True):
	print(token, end="")
pp(history)

Thinking Process:

1.  **Analyze the Request:** The user is asking for another number, following up on my previous response ("42").
2.  **Determine the appropriate response:** I need to provide a number. I can choose a random one, or a sequence, or a different kind of number. Since the previous one was "42" (often associated with Hitchhiker's Guide to the Galaxy), picking another random integer is simple and direct.
3.  **Select a number:** Let's pick 7. It's a common number, distinct from 42.
4.  **Format the output:** Just the number is sufficient, or a brief sentence. Given the brevity of the first response, keeping it brief is consistent.
5.  **Final Decision:** Output "7".
</think>

7
[{'role': 'system',
  'content': "You are a general-purpose assistant bot. Don't overthink."},
 {'role': 'user', 'content': 'Give me a number.'},
 {'role': 'assistant',
  'content': 'Thinking Process:\n'
             '\n'
             '1.  **Analyze the Request:**\n'
             '    *   User: "Give

In [52]:
for token in prompt(
		"Multiply those previous numbers together.",
		history=history,
		enable_thinking=True):
	print(token, end="")
pp(history)

Thinking Process:

1.  **Analyze the Request:** The user wants me to multiply the previous numbers I gave them.
2.  **Identify Previous Numbers:**
    *   First number: 42
    *   Second number: 7
3.  **Perform Calculation:** 42 * 7 = 294.
4.  **Formulate Output:** State the result clearly.
5.  **Final Check:** 40*7=280, 2*7=14, 280+14=294. Correct.

Output: 294.cw
</think>

294
[{'role': 'system',
  'content': "You are a general-purpose assistant bot. Don't overthink."},
 {'role': 'user', 'content': 'Give me a number.'},
 {'role': 'assistant',
  'content': 'Thinking Process:\n'
             '\n'
             '1.  **Analyze the Request:**\n'
             '    *   User: "Give me a number."\n'
             '    *   Intent: The user wants a single numerical value.\n'
             '    *   Constraints: None specified (e.g., range, type, '
             'context).\n'
             '\n'
             '2.  **Determine the Output:**\n'
             '    *   A simple number is sufficient.\n'
     

In [54]:
for token in prompt(
		"Explain Burnside’s Lemma. I am a first year undergraduate math student.",
		history=history,
		max_new_tokens=8192,
		enable_thinking=True):
	print(token, end="")

Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Burnside's Lemma.
    *   Target Audience: First-year undergraduate math student.
    *   Goal: Explain the concept clearly, intuitively, and with enough rigor to be useful but not overwhelming.

2.  **Determine the Key Concepts:**
    *   What is it? A theorem in combinatorics/group theory.
    *   What does it count? Orbits (distinct colorings/arrangements under symmetry).
    *   What are the ingredients? A finite group $G$, a set $X$ (objects), and the action of $G$ on $X$.
    *   What is the formula? $|X/G| = \frac{1}{|G|} \sum_{g \in G} |X^g|$, where $X^g$ is the set of elements fixed by $g$.
    *   Why is it useful? Counting distinct necklaces, patterns, etc., where symmetries matter.
    *   Analogy/Intuition: Averaging over symmetries.

3.  **Drafting the Explanation (Iterative Process):**

    *   *Introduction:* Start with the motivation. Counting things up to symmetry. Example: Colorings of a necklace.
    *  

In [69]:
os.makedirs(".data", exist_ok=True)
with open(".data/burnside.txt", "w") as f:
	f.write(history[-1]["content"])